In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [4]:
import wandb 
wandb.login(key=WB_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [11]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F

class ScratchMCQSolver(nn.Module): 
    def __init__(self, input_dim, hidden_dim=128): 
        super(ScratchMCQSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 5)

    def forward(self, x): 
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits 
vocab_size = 5000 
model_scratch = ScratchMCQSolver(input_dim=vocab_size)
print(model_scratch)

ScratchMCQSolver(
  (fc1): Linear(in_features=5000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=5, bias=True)
)


In [12]:
import pandas as pd 
import numpy as np 
import re 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity 

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [13]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [14]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [15]:
train_df.fillna("None", inplace=True)

In [16]:
def clean_text(text): 
    text = str(text).lower()
    text = re.sub(r'\s{2,}', '', text)
    return text.strip()

columns_to_clean = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in columns_to_clean: 
    if col in train_df.columns: 
        train_df[col] = train_df[col].apply(clean_text)

print("Data Cleaned")

Data Cleaned


In [17]:
vectorizer = TfidfVectorizer(stop_words='english')
predictions = []
actuals = []

for idx, row in train_df.iterrows():
    corpus = [row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]

    tfidf_matrix = vectorizer.fit_transform(corpus)
    prompt_vector = tfidf_matrix[0:1]
    options_matrix = tfidf_matrix[1:]

    similarities = cosine_similarity(prompt_vector, options_matrix).flatten()

    labels = ['A', 'B', 'C', 'D', 'E']
    top_3_idx = similarities.argsort()[-3:][::-1]
    top_3_preds = [labels[i] for i in top_3_idx]

    predictions.append(top_3_preds)
    if 'answer' in train_df.columns: 
        actuals.append(row['answer'])


In [18]:
def apk(actual, predicted, k=3): 
    predicted = predicted[:k]
    if actual in predicted: 
        return 1 / (predicted.index(actual)+1)

    return 0

if actuals: 
    map_3_score = np.mean([apk(a, p) for a, p in zip(actuals, predictions)])
    print(f'Baseline TF-IDF MAP@3: {map_3_score: .4f}')
else: 
    print('No answer column found')

Baseline TF-IDF MAP@3:  0.3260


# Mile 2

In [1]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df_pandas=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')
dataset = Dataset.from_pandas(df_pandas)

In [3]:
def combine_text_fn(example): 
    return {"combined_text": f"{example['prompt']} {example['A']}"}

dataset = dataset.map(combine_text_fn)

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_prompts(examples): 
    return tokenizer(examples['prompt'], padding='max_length', truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_prompts, batched=True)
input_ids_shape = np.array(tokenized_dataset['input_ids']).shape

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
model = AutoModel.from_pretrained('bert-base-uncased')
row_0_prompt = dataset[0]['prompt']
inputs_0 = tokenizer(row_0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs_0 = model(**inputs_0)

last_hidden_state = outputs_0.last_hidden_state 

cls_vector = last_hidden_state[0, 0, :5].tolist()

model_att = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
text_att = 'Light-ion fusion is a technique.'
inputs_att = tokenizer(text_att, return_tensors='pt')
tokens_att = tokenizer.convert_ids_to_tokens(inputs_att['input_ids'][0])

fusion_idx = tokens_att.index('fusion')

with torch.no_grad(): 
    outputs_att = model_att(**inputs_att)

attention_matrix = outputs_att.attentions[-1][0, 0]
weight_cls_to_fusion = attention_matrix[0, fusion_idx].item()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb_prompt = st_model.encode(dataset[0]['prompt'], convert_to_tensor=True)
emb_opt_b = st_model.encode(dataset[0]['B'], convert_to_tensor=True)
sim_score = util.cos_sim(emb_prompt, emb_opt_b).item()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def apk(actual, predicted, k=3): 
    predicted = predicted[:k]
    if actual in predicted: 
        return 1 / (predicted.index(actual) + 1)
    return 0 
labels = ['A', 'B', 'C', 'D', 'E']
tf_idf_vectorizer = TfidfVectorizer(stop_words='english')

minilm_preds = []
tfidf_preds = []
actual_answers = df_pandas['answer'].tolist() if 'answer' in df_pandas.columns else []

all_prompts = df_pandas['prompt'].tolist()
opt_cols = [df_pandas[c].tolist() for c in labels]

prompt_embs = st_model.encode(all_prompts, convert_to_tensor=True)
opt_embs = [st_model.encode(col, convert_to_tensor=True) for col in opt_cols]

for idx, row in df_pandas.iterrows(): 
    corpus = [row['prompt']] + [row[l] for l in labels]
    try: 
        tfidf_mat = tf_idf_vectorizer.fit_transform(corpus)
        sims_tfidf = cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:]).flatten()
        top_3_tfidf = [labels[i] for i in sims_tfidf.argsort()[-3:][::-1]]

    except: 
        top_3_tfidf = ['A', 'B', 'C']
    tfidf_preds.append(top_3_tfidf)

    p_emb = prompt_embs[idx]
    sims_minilm = []
    for o_idx in range(5): 
        sims_minilm.append(util.cos_sim(p_emb, opt_embs[o_idx][idx]).item())

    top_3_minilm = [labels[i] for i in np.argsort(sims_minilm)[-3:][::-1]]
    minilm_preds.append(top_3_minilm)

map3_minilm = np.mean([apk(a, p) for a, p in zip(actual_answers, minilm_preds)])

improvement_count = 0
for a, t_pred, m_pred in zip(actual_answers, tfidf_preds, minilm_preds): 
    if(a in m_pred) and (a not in t_pred): 
        improvement_count += 1

In [8]:
classifier = pipeline('zero-shot-classification', model = 'facebook/bart-large-mnli', device=0 if torch.cuda.is_available() else -1)
prompt_idx_1 = dataset[1]['prompt']
candidates = [dataset[1]['A'], dataset[1]['B'], dataset[1]['C']]

res_softmax = classifier(prompt_idx_1, candidate_labels = candidates, multi_label=False)
top_prob_softmax = res_softmax['scores'][0]

res_sigmoid = classifier(prompt_idx_1, candidate_labels=candidates, multi_label=True)
abs_diff = abs(sum(res_softmax['scores']) - sum(res_sigmoid['scores']))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [15]:
from transformers import AutoModelForSeq2SeqLM

In [16]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
slm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to("cuda:0")
prompt_slm = f"Question: {dataset[0]['prompt']}. Is the correct answer A: {dataset[0]['A']} or B: {dataset[0]['B']}? Answer with just the letter A or B."
inputs = tokenizer(prompt_slm, return_tensors="pt").to("cuda:0")
outputs = slm_model.generate(**inputs, max_new_tokens=5)
slm_out = tokenizer.decode(outputs[0], skip_special_tokens=True)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Answers Milstone 2

In [17]:
print(f"Q1: Character length at index 51: {len(dataset[51]['combined_text'])}")
print(f"Q2: Vocabulary Size: {tokenizer.vocab_size}")
print(f"Q3: [SEP] Token ID: {tokenizer.convert_tokens_to_ids('[SEP]')}")
print(f"Q4: Geometric shape of input_ids tensor: {input_ids_shape}")
print(f"Q5: Dimensionality of each individual attention head: {768 // 12}")
print(f"Q6: Shape of last_hidden_state tensor: {list(last_hidden_state.shape)}")
print(f"Q7: Sum of first 5 float values in [CLS] vector: {round(sum(cls_vector), 4)}")
print(f"Q8: Attention weight from [CLS] to 'fusion': {round(weight_cls_to_fusion, 4)}")
print(f"Q9: Cosine similarity between prompt and Option B: {round(sim_score, 4)}")
print(f"Q10 Part 1: Final MAP@3 score of MiniLM pipeline: {round(map3_minilm, 4)}")
print(f"Q10 Part 2: Number of questions saved by MiniLM: {improvement_count}")

Q1: Character length at index 51: 614
Q2: Vocabulary Size: 32100
Q3: [SEP] Token ID: 2
Q4: Geometric shape of input_ids tensor: (2000, 128)
Q5: Dimensionality of each individual attention head: 64
Q6: Shape of last_hidden_state tensor: [1, 31, 768]
Q7: Sum of first 5 float values in [CLS] vector: -1.2001
Q8: Attention weight from [CLS] to 'fusion': 0.1025
Q9: Cosine similarity between prompt and Option B: 0.7658
Q10 Part 1: Final MAP@3 score of MiniLM pipeline: 0.4231
Q10 Part 2: Number of questions saved by MiniLM: 488
